# AirCanopy: wind solver on a graphics card

This notebook runs the parts of phase P2 that are too big for a processor (JOURNAL.md, Steps 10 to 12):

1. **Speed** of the wind solver on this graphics card (`benchmarks/wind_speed.py`, 128³ and 256³ cells).
2. **The 3D Taylor–Green vortex at Reynolds number 1600** on 128³ and 256³ cells, the first test with real turbulence (`benchmarks/wind_tgv1600.py`): D3Q27 with the recursive collision, the solver's default since Step 12, and D3Q19 with the regularised collision for comparison (its 128³ run also checks the one on the processor).

**Before you start:** *Runtime → Change runtime type → GPU* (an A100 is best; an L4 or T4 works, more slowly). Then *Runtime → Run all*. How long it takes depends on the card: the speed step prints the cell updates per second it measured, and each Taylor–Green run on 256³ cells needs about 2.4 × 10¹¹ of them (16.8 million cells × 14 100 steps).

**At the end** a file `aircanopy_gpu_results.zip` downloads. Upload it to the Claude session; it will be checked and committed there.

In [ ]:
# 1. Which graphics card do we have?
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## Get the code

The repository is cloned at the latest `main`. If the repository is private, add a GitHub token (read access is enough) as a Colab secret called `GITHUB_TOKEN` (key icon on the left), and allow this notebook to use it.

In [ ]:
import os
import subprocess

repo = "github.com/chingkheinganba231005/AirCanopy.git"
url = f"https://{repo}"
try:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
    if token:
        url = f"https://{token}@{repo}"
except Exception:
    pass  # no secret: the repository is public
if not os.path.isdir("aircanopy"):
    subprocess.run(["git", "clone", "--depth", "1", url, "aircanopy"], check=True)
%cd aircanopy
!git log -1 --format='%H %s'

## Install

`uv` installs exactly the versions in `uv.lock`, in a separate environment (`.venv`), so Colab's own packages are not touched. Then JAX's CUDA support is added to that environment, held to releases at least two weeks old like everything else (D-009).

In [ ]:
!pip install -q uv
!uv sync --locked
!uv pip install --python .venv "jax[cuda12]==0.11.2" --exclude-newer 2026-10-09T00:00:00Z
!.venv/bin/python -c "import jax; print(jax.__version__, jax.devices())"

## 1. Speed

Million cell updates per second (MLUPS) for D3Q19 with the regularised collision and D3Q27 with the regularised and the recursive one, in 32-bit and 64-bit numbers, on boxes of 128³ and 256³ cells. Adds the runs to `results/wind_speed_gpu.json`.

In [ ]:
!.venv/bin/python benchmarks/wind_speed.py 128 400
!.venv/bin/python benchmarks/wind_speed.py 256 200

## 2. Taylor–Green vortex at Re 1600

128³ and 256³ cells with no turbulence model: D3Q27 with the recursive collision, then D3Q19 with the regularised one (its 128³ run compares with the processor's). The pass criteria were written down before the first run: the peak of the dissipation rate within 5 % of the reference, at a time within 0.5 time units of it (on 256³). Writes `results/wind_tgv1600_gpu.json`.

In [ ]:
!.venv/bin/python benchmarks/wind_tgv1600.py 128 256
!.venv/bin/python benchmarks/wind_tgv1600.py 128 256 --lattice D3Q19

## Pack the results

The zip holds the result files and a manifest: the commit that ran, the graphics card, and the time.

In [ ]:
import datetime
import hashlib
import json
import pathlib
import subprocess
import zipfile

from google.colab import files as colab_files


def output(*command: str) -> str:
    return subprocess.run(command, capture_output=True, text=True).stdout.strip()


names = ["results/wind_speed_gpu.json", "results/wind_tgv1600_gpu.json"]
files = [p for p in names if pathlib.Path(p).is_file()]
manifest = {
    "commit": output("git", "rev-parse", "HEAD"),
    "gpu": output(
        "nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"
    ),
    "packed_at": datetime.datetime.now(datetime.UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "files": {p: hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest() for p in files},
}
with zipfile.ZipFile("aircanopy_gpu_results.zip", "w") as z:
    for p in files:
        z.write(p)
    z.writestr("manifest.json", json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
colab_files.download("aircanopy_gpu_results.zip")